In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, 
                             roc_curve, mean_squared_error, r2_score)

# Gradient Boosting 계열 고도화 라이브러리
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import shap  # SHAP 분석 패키지

warnings.filterwarnings('ignore')

# ==============================================================================
# [단계 0] 환경 설정 및 고도화 전처리 데이터 로드
# ==============================================================================
print("="*70)
print("[단계 0] 환경 설정 및 새로 개편된 고도화 데이터셋 로드")
print("="*70)

os.makedirs('figs_advanced', exist_ok=True)
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

# 전처리 데이터 로드
df = pd.read_csv('./dataset/processed_ml_dataset.csv', encoding='utf-8-sig')

# 피처 추출 (기본 피처 + 핵심 파생 피처)
feature_cols = [
    'min_temp', 'max_temp', 'avg_temp', 'avg_rhm', 'annual_rn',
    'opt_temp_min', 'opt_temp_max', 'frost_limit_temp', 'opt_humidity',
    'soil_ph_min', 'soil_ph_max',
    # 핵심 개선 파생 피처
    'temp_diff_from_opt', 'frost_safety_margin'
]

# 사용 가능한 피처 체크 및 선택
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()

# 분류 타깃 (0/1 적합 여부) 및 연속형 점수 타깃 선택
y_cls = df['suitability'] if 'suitability' in df.columns else (df['suitability_score'] >= 65).astype(int)
y_reg = df['suitability_score'] if 'suitability_score' in df.columns else None

print(f"✔ 데이터 로드 완료: {X.shape[0]}행 {X.shape[1]}개 피처")
print(f"✔ 타깃 분포 (0: 부적합, 1: 적합):\n{y_cls.value_counts(normalize=True).round(3)}")


# ==============================================================================
# [단계 1] K-Fold 교차 검증 기반 모델 벤치마킹 (실패/성공 비교 체계)
# ==============================================================================
print("\n" + "="*70)
print("[단계 1] 5-Fold Cross Validation 기반 모델 벤치마킹")
print("="*70)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 비교 모델군 정의
base_models = {
    'Baseline_DT': HistGradientBoostingClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, verbose=-1, random_state=42)
}

cv_results = []

for name, model in base_models.items():
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    scores = cross_validate(model, X, y_cls, cv=skf, scoring=scoring, n_jobs=-1)
    
    cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean()
    })
    print(f"  - [{name}] 5-Fold F1: {scores['test_f1'].mean():.4f} | ROC-AUC: {scores['test_roc_auc'].mean():.4f}")

df_benchmark = pd.DataFrame(cv_results).sort_values(by='F1-Score', ascending=False)


# ==============================================================================
# [단계 2] 하이브리드 스태킹(Hybrid Stacking) 모델 구축 및 학습
# ==============================================================================
print("\n" + "="*70)
print("[단계 2] 하이브리드 스태킹(Hybrid Stacking) 앙상블 구축")
print("="*70)

# 1계층 개별 모델 (Base Estimators)
estimators = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, eval_metric='logloss', random_state=42)),
    ('lgb', LGBMClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, verbose=-1, random_state=42))
]

# 메타 모델 (Final Estimator)
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=1.0, random_state=42),
    cv=5,
    n_jobs=-1
)

# 스태킹 교차 검증
stack_scores = cross_validate(stacking_model, X, y_cls, cv=skf, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])

stack_result = {
    'Model': 'Hybrid_Stacking',
    'Accuracy': stack_scores['test_accuracy'].mean(),
    'Precision': stack_scores['test_precision'].mean(),
    'Recall': stack_scores['test_recall'].mean(),
    'F1-Score': stack_scores['test_f1'].mean(),
    'ROC-AUC': stack_scores['test_roc_auc'].mean()
}

print(f"  - [Hybrid_Stacking] 5-Fold F1: {stack_result['F1-Score']:.4f} | ROC-AUC: {stack_result['ROC-AUC']:.4f}")

df_benchmark = pd.concat([df_benchmark, pd.DataFrame([stack_result])], ignore_index=True).sort_values(by='F1-Score', ascending=False)


# ==============================================================================
# [단계 3] 최종 모델 검증 시각화 (보고서/사이트 제출용 차트 생성)
# ==============================================================================
print("\n" + "="*70)
print("[단계 3] 보고서용 심층 평가 및 시각화 차트 생성")
print("="*70)

# 전체 데이터 학습 및 최종 모델 추출
stacking_model.fit(X, y_cls)

# --- [시각화 1] 모델별 F1-Score & ROC-AUC 비교 차트 ---
plt.figure(figsize=(9, 5))
sns.barplot(data=df_benchmark, x='Model', y='F1-Score', palette='Blues_r')
plt.title('5-Fold Cross Validation F1-Score Comparison', fontsize=14, fontweight='bold')
plt.ylim(0.7, 1.0)
for idx, row in df_benchmark.reset_index().iterrows():
    plt.text(idx, row['F1-Score'] + 0.005, f"{row['F1-Score']:.3f}", ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('figs_advanced/01_model_benchmark.png', dpi=150)
plt.close()
print("✔ [1/3] 모델 벤치마킹 비교 차트 저장 완료: figs_advanced/01_model_benchmark.png")

# --- [시각화 2] SHAP (Shapley Additive exPlanations) 설명 가능성 분석 ---
# LightGBM 단일 모델 기준으로 피처 영향력 세부 분석
lgb_model = LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, verbose=-1, random_state=42)
lgb_model.fit(X, y_cls)

explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, show=False)
plt.title('SHAP Feature Importance & Impact Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figs_advanced/02_shap_summary.png', dpi=150)
plt.close()
print("✔ [2/3] SHAP 설명 가능성 분석 차트 저장 완료: figs_advanced/02_shap_summary.png")

# --- [시각화 3] 혼동 행렬 (Confusion Matrix) ---
y_pred_final = stacking_model.predict(X)
cm = confusion_matrix(y_cls, y_pred_final)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('Final Hybrid Stacking Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.savefig('figs_advanced/03_confusion_matrix.png', dpi=150)
plt.close()
print("✔ [3/3] 최종 혼동 행렬 차트 저장 완료: figs_advanced/03_confusion_matrix.png")



[단계 0] 환경 설정 및 새로 개편된 고도화 데이터셋 로드
✔ 데이터 로드 완료: 9800행 13개 피처
✔ 타깃 분포 (0: 부적합, 1: 적합):
suitability
0    0.966
1    0.034
Name: proportion, dtype: float64

[단계 1] 5-Fold Cross Validation 기반 모델 벤치마킹
  - [Baseline_DT] 5-Fold F1: 0.9836 | ROC-AUC: 0.9999
  - [RandomForest] 5-Fold F1: 0.9487 | ROC-AUC: 0.9999
  - [XGBoost] 5-Fold F1: 0.9852 | ROC-AUC: 0.9999
  - [LightGBM] 5-Fold F1: 0.9805 | ROC-AUC: 0.9999

[단계 2] 하이브리드 스태킹(Hybrid Stacking) 앙상블 구축
  - [Hybrid_Stacking] 5-Fold F1: 0.9803 | ROC-AUC: 1.0000

[단계 3] 보고서용 심층 평가 및 시각화 차트 생성
✔ [1/3] 모델 벤치마킹 비교 차트 저장 완료: figs_advanced/01_model_benchmark.png
✔ [2/3] SHAP 설명 가능성 분석 차트 저장 완료: figs_advanced/02_shap_summary.png
✔ [3/3] 최종 혼동 행렬 차트 저장 완료: figs_advanced/03_confusion_matrix.png

[단계 4] 사이트온/발표 보고용 모델 성능 요약
          Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC
        XGBoost  0.998980   0.979750 0.991045  0.985247 0.999935
    Baseline_DT  0.998878   0.985158 0.982177  0.983613 0.999950
       LightGBM  0.998673   0.985022 0.976